In [ ]:
import os, tensorflow as tf
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    print("TPU available!")
    BATCH_SIZE = 128
except (ValueError, tf.errors.NotFoundError):
    print("No TPU. Using GPU/CPU.")
    gpus = tf.config.list_physical_devices("GPU")
    strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
    BATCH_SIZE = 32
    !nvidia-smi


In [ ]:
!pip install -q opendatasets
import opendatasets as od
import os

dataset_url = "https://www.kaggle.com/datasets/belalsafy/egyptian-new-currency-2023"
od.download(dataset_url)

data_dir = "./egyptian-new-currency-2023/dataset"
print(f"Checking path: {data_dir}")
if os.path.exists(data_dir):
    print("Dataset path verified.")

In [ ]:
import os
import tensorflow as tf

valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.gif')
data_dir = './egyptian-new-currency-2023/dataset/train'

print("Verifying image integrity...")
removed_count = 0

for root, dirs, files in os.walk(data_dir):
    for file in files:
        file_path = os.path.join(root, file)
        if not file.lower().endswith(valid_extensions):
            try:
                os.remove(file_path)
                removed_count += 1
                continue
            except: pass
        try:
            img_bytes = tf.io.read_file(file_path)
            tf.io.decode_image(img_bytes)
        except Exception:
            print(f"Removing corrupted file: {file_path}")
            try:
                os.remove(file_path)
                removed_count += 1
            except: pass

print(f"Cleanup complete. Removed {removed_count} problematic files.")

## RUN 4: REVERT TO RUN 2 CONFIG + MORE AUGMENTATION
### Fix from Run 3 Catastrophe:
- REVERT to Run 2 architecture (no L2!)
- Keep LR at 1e-4 (NOT 5e-5)
- Add more augmentation only
- Same dropout rates as Run 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import datetime

RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./visualizations/run4_{RUN_TIMESTAMP}"
os.makedirs(RUN_DIR, exist_ok=True)

data_dir = "./egyptian-new-currency-2023/dataset"
train_dir = os.path.join(data_dir, "train")
IMG_HEIGHT = 224
IMG_WIDTH = 224

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="training",
    seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.2, subset="validation",
    seed=123, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print(f"Classes: {class_names}")
print(f"Number of classes: {NUM_CLASSES}")
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


In [ ]:
print("Run 4 already completed previously. Skipping to Run 5.")


In [ ]:
NUM_RUNS = 10
EPOCHS = 30  # More epochs

best_overall_acc = 0
best_overall_run = 0
best_overall_model = None
best_history = None
all_run_metrics = []

print(f"{'='*60}")
print(f"STARTING {NUM_RUNS} TRAINING RUNS (Run 4)")
print(f"Config: REVERT to Run 2 + more augmentation")
print(f"LR: 1e-4 (NOT 5e-5 like Run 3!)")
print(f"NO L2 regularization!")
print(f"{'='*60}\n")

for run_num in range(1, NUM_RUNS + 1):
    print(f"{'='*60}")
    print(f"RUN {run_num}/{NUM_RUNS}")
    print(f"{'='*60}")
    
    model = create_model()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=initial_lr),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=["accuracy"])
    
    EARLY_STOPPING = callbacks.EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=0)
    REDUCE_LR = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=0)
    
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[EARLY_STOPPING, REDUCE_LR], verbose=0)
    
    val_acc = history.history["val_accuracy"]
    train_acc = history.history["accuracy"]
    
    best_val_acc = max(val_acc)
    best_epoch = val_acc.index(best_val_acc) + 1
    final_val_acc = val_acc[-1]
    final_train_acc = train_acc[-1]
    epochs_run = len(val_acc)
    
    run_metrics = {"run": run_num, "best_val_acc": best_val_acc, "best_epoch": best_epoch, "final_val_acc": final_val_acc, "final_train_acc": final_train_acc, "epochs_run": epochs_run}
    all_run_metrics.append(run_metrics)
    
    print(f"Run {run_num}: Best Val Acc = {best_val_acc:.4f} (epoch {best_epoch})")
    print(f"         Final Val Acc = {final_val_acc:.4f}, Train Acc = {final_train_acc:.4f}")
    
    if best_val_acc > best_overall_acc:
        best_overall_acc = best_val_acc
        best_overall_run = run_num
        best_overall_model = model
        best_history = history
        print(f"         *** NEW BEST! ***")
    
    if run_num % 2 == 0:
        print(f"\n>>> PROGRESS (Run {run_num}/{NUM_RUNS}) <<<")
        for m in all_run_metrics:
            print(f"    Run {m.get('run')}: Best={m.get('best_val_acc'):.4f}, Final={m.get('final_val_acc'):.4f}")
        print(f"    >> Best: {best_overall_acc:.4f} (Run {best_overall_run})")
        print(f"    >> Target: 0.93 | Gap: {0.93 - best_overall_acc:.4f}\n")

print(f"\n{'='*60}")
print(f"ALL {NUM_RUNS} RUNS COMPLETED")
print(f"{'='*60}")
print(f"Best validation accuracy: {best_overall_acc:.4f} (Run {best_overall_run})")
print(f"Target: 0.93 (93%)")
print(f"Gap to target: {0.93 - best_overall_acc:.4f}")

In [ ]:
run_nums = [m.get('run') for m in all_run_metrics]
best_vals = [m.get('best_val_acc') for m in all_run_metrics]
final_vals = [m.get('final_val_acc') for m in all_run_metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(run_nums, best_vals, 'b-o', label='Best Val Acc', linewidth=2, markersize=8)
axes[0].plot(run_nums, final_vals, 'r--s', label='Final Val Acc', linewidth=2, markersize=8)
axes[0].axhline(y=0.93, color='g', linestyle=':', linewidth=2, label='Target: 93%')
axes[0].set_xlabel('Run Number', fontsize=12)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].set_title(f'All 10 Runs (Run 4)', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(1, 11))

axes[1].bar(run_nums, best_vals, color='steelblue', alpha=0.7)
axes[1].axhline(y=0.93, color='r', linestyle='--', linewidth=2, label='Target: 93%')
axes[1].set_xlabel('Run Number', fontsize=12)
axes[1].set_ylabel('Best Validation Accuracy', fontsize=12)
axes[1].set_title('Best Accuracy per Run', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(f"{RUN_DIR}/run_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nBest overall: {best_overall_acc:.4f} (Run {best_overall_run})")
print(f"Target: 0.93 | Gap: {0.93 - best_overall_acc:.4f}")

In [ ]:
acc = best_history.history["accuracy"]
val_acc = best_history.history["val_accuracy"]
loss = best_history.history["loss"]
val_loss = best_history.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc, label="Training Accuracy", linewidth=2)
axes[0].plot(val_acc, label="Validation Accuracy", linewidth=2)
axes[0].axhline(y=0.93, color="g", linestyle="--", label="Target: 93%")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title(f"Accuracy Curves (Best Run: {best_overall_run})", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss, label="Training Loss", linewidth=2)
axes[1].plot(val_loss, label="Validation Loss", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss", fontsize=12)
axes[1].set_title("Loss Curves", fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RUN_DIR}/accuracy_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n--- Classification Report ---")
y_true = np.concatenate([y for x, y in val_ds], axis=0)
y_pred = np.argmax(best_overall_model.predict(val_ds), axis=-1)
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
best_overall_model.save(f"{RUN_DIR}/best_model.keras")

run_results = []
for m in all_run_metrics:
    run_results.append(f"Run {m.get('run')}: Best={m.get('best_val_acc'):.4f} (ep{m.get('best_epoch')}), Final={m.get('final_val_acc'):.4f}")

result_text = "\n".join(run_results)

best_val_acc = best_overall_acc

run4_metrics = f"""### Run 4 (REVERT to Run 2 + More Aug)
**Date**: {RUN_TIMESTAMP}
**Status**: {'COMPLETED - Target Met!' if best_val_acc >= 0.93 else 'COMPLETED - Below Target'}

### Configuration
- Architecture: Run 2 config (4 Conv + BatchNorm, NO L2!)
- Augmentation: Flip, Rotation(0.2), Zoom(0.2), + Brightness(0.1), Contrast(0.1)
- Optimizer: Adam (lr=1e-4) - SAME AS RUN 2, NOT 5e-5!
- Dropout: 0.2, 0.3, 0.5 (same as Run 2)
- Epochs per run: {EPOCHS}
- Number of runs: {NUM_RUNS}

### Results (All {NUM_RUNS} Runs)
{result_text}

### Best Result
- **Best Validation Accuracy**: {best_val_acc:.4f} (Run {best_overall_run})
- **Target**: 0.93 (93%)
- **Gap**: {0.93 - best_val_acc:.4f}
- Target Met: {'YES ✓' if best_val_acc >= 0.93 else 'NO'}

---
"""

with open("RUN_TRACKING.md", "a") as f:
    f.write(run4_metrics)

print(f"\nAll run metrics saved to RUN_TRACKING.md")
print(f"Visualizations saved to {RUN_DIR}")
print(f"Best model saved to {RUN_DIR}/best_model.keras")

## RUN 5: IMPROVED ARCHITECTURE TARGETING 93%
### Config:
- 2 Conv layers per block (deeper): 32->64->128->256
- TPU-optimized: 5 sub-runs x 30 epochs, batch_size=128 on TPU
- SGD+momentum + CosineDecay LR (0.01 -> 1e-5)
- Label smoothing (0.1)
- Ensemble top 3 runs


In [ ]:
import datetime
RUN5_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN5_DIR = f"./visualizations/run5_{RUN5_TIMESTAMP}"
os.makedirs(RUN5_DIR, exist_ok=True)
print(f"Run 5 outputs -> {RUN5_DIR}")
print(f"Classes: {class_names}")
print(f"Train: {len(train_ds)} batches, Val: {len(val_ds)} batches")

In [ ]:
def create_model_v5(num_classes):
    data_aug = models.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.3),
        layers.RandomZoom(0.2),
        layers.RandomBrightness(0.15),
        layers.RandomContrast(0.15),
        layers.RandomTranslation(0.1, 0.1),
    ])
    model = models.Sequential([
        data_aug,
        layers.Rescaling(1./255),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.35),
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Dropout(0.5),
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

initial_lr = 0.01
decay_steps = 1000
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=initial_lr, decay_steps=decay_steps, alpha=1e-5/initial_lr
)

model_v5 = create_model_v5(NUM_CLASSES)
model_v5.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, label_smoothing=0.1),
    metrics=["accuracy"],
)
print("\n--- Run 5 Architecture (Target: 93%) ---")
model_v5.summary()

In [ ]:
NUM_RUNS_5 = 5
EPOCHS_5 = 30

best5_acc = 0
best5_run = 0
best5_model = None
best5_history = None
all5_metrics = []
top3_models = []

print(f"{'='*60}")
print(f"RUN 5: STARTING {NUM_RUNS_5} TRAINING RUNS")
print(f"Architecture: Deeper 2-conv blocks, SGD+Momentum+CosineDecay")
print(f"Label smoothing: 0.1 | Epochs: {EPOCHS_5} per run")
print(f"Target: 93% | Current best from Run 4: 69.01%")
print(f"{'='*60}\n")

for run_num in range(1, NUM_RUNS_5 + 1):
    print(f"{'='*60}")
    print(f"RUN 5 - RUN {run_num}/{NUM_RUNS_5}")
    print(f"{'='*60}")

    with strategy.scope():
            m = create_model_v5(NUM_CLASSES)
        m.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=lr_schedule, momentum=0.9),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, label_smoothing=0.1),
        metrics=["accuracy"],
    )

    es = callbacks.EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True, verbose=0)
    rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6, verbose=0)
    cp = callbacks.ModelCheckpoint(f"{RUN5_DIR}/run{run_num}_best.keras", monitor="val_accuracy", save_best_only=True, verbose=0)

    history = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_5, callbacks=[es, rlr, cp], verbose=0)

    val_acc = history.history["val_accuracy"]
    train_acc = history.history["accuracy"]
    best_val = max(val_acc)
    best_ep = val_acc.index(best_val) + 1
    final_val = val_acc[-1]
    final_train = train_acc[-1]
    epochs_run = len(val_acc)

    metrics = {"run": run_num, "best": best_val, "epoch": best_ep, "final_val": final_val, "final_train": final_train, "epochs": epochs_run}
    all5_metrics.append(metrics)

    marker = ""
    if best_val > best5_acc:
        best5_acc = best_val
        best5_run = run_num
        best5_model = m
        best5_history = history
        marker = " <<< BEST"
    if best_val >= 0.93:
        marker += " *** TARGET MET! ***"

    print(f"  Best={best_val:.4f} (ep{best_ep})  Final={final_val:.4f}  Train={final_train:.4f}{marker}")

    top3_models.append((best_val, m))
    top3_models.sort(key=lambda x: -x[0])
    top3_models = top3_models[:3]

    if run_num % 2 == 0 or run_num == NUM_RUNS_5:
        print(f"\n>>> PROGRESS (Run {run_num}/{NUM_RUNS_5}) <<<")
        for m2 in all5_metrics:
            print(f"  Run {m2.get('run')}: Best={m2['best']:.4f} (ep{m2['epoch']})")
        print(f"  >> Current Best: {best5_acc:.4f} (Run {best5_run})")
        print(f"  >> Target: 0.93 | Gap: {0.93 - best5_acc:.4f}")
        print(f"  >> Ensemble (top 3): {[x[0]:.4f for x in top3_models]}\n")

print(f"\n{'='*60}")
print(f"RUN 5: ALL {NUM_RUNS_5} RUNS COMPLETED")
print(f"{'='*60}")
print(f"Best individual: {best5_acc:.4f} (Run {best5_run})")
print(f"Top 3 ensemble: {[x[0]:.4f for x in top3_models]}")
print(f"Target: 0.93 | Gap: {0.93 - best5_acc:.4f}")

    # auto-save progress
    import json as _js
    with open("run5_progress.json", "w") as _f:
        _js.dump({"best_acc": best5_acc, "best_run": best5_run, "runs": all5_metrics, "current_run": run_num}, _f)


In [ ]:
import numpy as np

run_nums5 = [m.get('run') for m in all5_metrics]
best5_vals = [m.get('best') for m in all5_metrics]
final5_vals = [m.get('final_val') for m in all5_metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(run_nums5, best5_vals, 'b-o', label='Best Val Acc', linewidth=2, markersize=8)
axes[0].plot(run_nums5, final5_vals, 'r--s', label='Final Val Acc', linewidth=2, markersize=8)
axes[0].axhline(y=0.93, color='g', linestyle=':', linewidth=2, label='Target: 93%')
axes[0].set_xlabel('Run Number', fontsize=12)
axes[0].set_ylabel('Validation Accuracy', fontsize=12)
axes[0].set_title('Run 5: All 10 Runs', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(1, 11))

axes[1].bar(run_nums5, best5_vals, color='steelblue', alpha=0.7)
axes[1].axhline(y=0.93, color='r', linestyle='--', linewidth=2, label='Target: 93%')
axes[1].axhline(y=best5_acc, color='darkorange', linestyle='-', linewidth=2, label=f'Best: {best5_acc:.4f}')
axes[1].set_xlabel('Run Number', fontsize=12)
axes[1].set_ylabel('Best Validation Accuracy', fontsize=12)
axes[1].set_title('Best Accuracy per Run', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best: {best5_acc:.4f} (Run {best5_run}) | Target: 0.93 | Gap: {0.93 - best5_acc:.4f}")

In [ ]:
acc5 = best5_history.history["accuracy"]
val_acc5 = best5_history.history["val_accuracy"]
loss5 = best5_history.history["loss"]
val_loss5 = best5_history.history["val_loss"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(acc5, label="Training Accuracy", linewidth=2)
axes[0].plot(val_acc5, label="Validation Accuracy", linewidth=2)
axes[0].axhline(y=0.93, color="g", linestyle="--", label="Target: 93%")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Accuracy", fontsize=12)
axes[0].set_title(f"Run 5 Accuracy Curves (Best Run: {best5_run})", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(loss5, label="Training Loss", linewidth=2)
axes[1].plot(val_loss5, label="Validation Loss", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Loss", fontsize=12)
axes[1].set_title("Run 5 Loss Curves", fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_accuracy_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n--- RUN 5 Classification Report (Best Model) ---")
y_true = np.concatenate([y for x, y in val_ds], axis=0)
y_pred_best = np.argmax(best5_model.predict(val_ds), axis=-1)
print(classification_report(y_true, y_pred_best, target_names=class_names))

cm = confusion_matrix(y_true, y_pred_best)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.title("Run 5 Confusion Matrix (Best Single Model)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RUN5_DIR}/run5_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n--- RUN 5 Ensemble Prediction (Top 3 Models) ---")
all_preds = []
for val, m in top3_models:
    p = m.predict(val_ds, verbose=0)
    all_preds.append(p)
    print(f"  Model (val_acc={val:.4f}): included")
avg_preds = np.mean(all_preds, axis=0)
y_pred_ensemble = np.argmax(avg_preds, axis=-1)
from sklearn.metrics import accuracy_score
ensemble_acc = accuracy_score(y_true, y_pred_ensemble)
print(f"\nEnsemble (top 3) Val Accuracy: {ensemble_acc:.4f}")
print(f"Best Single Model Val Accuracy: {best5_acc:.4f}")
improvement = ensemble_acc - best5_acc
print(f"Ensemble Improvement: +{improvement:.4f}")

if ensemble_acc >= 0.93:
    print(f"\n{'='*60}")
    print(f"*** TARGET 93% MET! Ensemble accuracy: {ensemble_acc:.4f} ***")
    print(f"{'='*60}")
else:
    print(f"\nTarget: 0.93 | Ensemble Gap: {0.93 - ensemble_acc:.4f}")

In [ ]:
best5_model.save(f"{RUN5_DIR}/best_model_run5.keras")
for i, (val, m) in enumerate(top3_models):
    m.save(f"{RUN5_DIR}/ensemble_model_{i+1}_{val:.4f}.keras")

run_results_5 = []
for m2 in all5_metrics:
    run_results_5.append(f"Run {m2.get('run')}: Best={m2['best']:.4f} (ep{m2['epoch']}), Final={m2['final_val']:.4f}")

ensemble_str = f"Ensemble (top 3): {[x[0]:.4f for x in top3_models]}"
result_text_5 = "\n".join(run_results_5 + [ensemble_str])

final_acc = max(best5_acc, ensemble_acc)

run5_metrics = f"""### Run 5 (Improved Architecture - Target 93%)
**Date**: {RUN5_TIMESTAMP}
**Status**: {'COMPLETED - TARGET MET! :rocket:' if final_acc >= 0.93 else 'COMPLETED - Below Target'}

### Configuration
- Architecture: 2 Conv/block (32→64→128→256), Dense(512→256) head
- Augmentation: Flip, Rotation(0.3), Zoom(0.2), Brightness(0.15), Contrast(0.15), Translation(0.1)
- Optimizer: SGD (momentum=0.9) + CosineDecay LR (0.01 → 1e-5)
- Loss: SparseCategoricalCrossentropy + LabelSmoothing(0.1)
- Dropout: 0.25, 0.35, 0.5, 0.5 (conv blocks), 0.5, 0.3 (dense)
- Epochs per run: {EPOCHS_5}
- Number of runs: {NUM_RUNS_5}

### Results (All {NUM_RUNS_5} Runs)
{result_text_5}

### Best Result
- **Best Validation Accuracy (Single Model)**: {best5_acc:.4f} (Run {best5_run})
- **Ensemble Accuracy (Top 3)**: {ensemble_acc:.4f}
- **Target**: 0.93 (93%)
- **Gap**: {0.93 - final_acc:.4f}
- Target Met: {'YES :rocket:' if final_acc >= 0.93 else 'NO'}

---
"""

with open("RUN_TRACKING.md", "a") as f:
    f.write(run5_metrics)

print(f"\nRun 5 metrics saved to RUN_TRACKING.md")
print(f"Visualizations saved to {RUN5_DIR}")
print(f"Best model saved to {RUN5_DIR}/best_model_run5.keras")
print(f"Ensemble models saved to {RUN5_DIR}/ensemble_model_*.keras")